In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

binary_classification_with_a_bank_dataset_clone_path = kagglehub.competition_download('binary-classification-with-a-bank-dataset-clone')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# 1. Imports
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

import warnings
warnings.filterwarnings("ignore")

# 2. Load data (Kaggle auto-mounts /kaggle/input)
train = pd.read_csv("/kaggle/input/binary-classification-with-a-bank-dataset-clone/train.csv")
test  = pd.read_csv("/kaggle/input/binary-classification-with-a-bank-dataset-clone/test.csv")

print(train.shape, test.shape)
train.head()


In [ ]:
# 3. Separate target and ID
TARGET = "y"        # adjust if the target column has a different name
ID_COL = "id"

y = train[TARGET]
train_id = train[ID_COL]
test_id  = test[ID_COL]

train_features = train.drop([TARGET, ID_COL], axis=1)
test_features  = test.drop([ID_COL], axis=1)

# 4. Identify categorical and numeric columns
cat_cols = train_features.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = train_features.select_dtypes(exclude=["object", "category"]).columns.tolist()

print("Categorical:", cat_cols)
print("Numeric    :", num_cols)

# 5. Simple label encoding for tree models (works fine for boosting)
from sklearn.preprocessing import OrdinalEncoder

if len(cat_cols) > 0:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    train_features[cat_cols] = oe.fit_transform(train_features[cat_cols])
    test_features[cat_cols]  = oe.transform(test_features[cat_cols])


In [ ]:
# 6. Stratified K-Fold
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

X = train_features.values
X_test = test_features.values
y_array = y.values

oof_cat = np.zeros(len(train))
oof_lgb = np.zeros(len(train))
oof_xgb = np.zeros(len(train))

preds_cat = np.zeros(len(test))
preds_lgb = np.zeros(len(test))
preds_xgb = np.zeros(len(test))


In [ ]:
# 7. Faster training with smaller models
for fold, (trn_idx, val_idx) in enumerate(skf.split(X, y_array)):
    print(f"Fold {fold+1}/{N_SPLITS}")

    X_tr, X_val = X[trn_idx], X[val_idx]
    y_tr, y_val = y_array[trn_idx], y_array[val_idx]

    # CatBoost (fewer iterations)
    model_cat = CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="AUC",
        depth=6,
        learning_rate=0.05,
        iterations=600,          # was 2000
        random_seed=42,
        verbose=False,
        od_type="Iter",
        od_wait=50
    )
    model_cat.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)
    val_pred_cat = model_cat.predict_proba(X_val)[:, 1]
    oof_cat[val_idx] = val_pred_cat
    preds_cat += model_cat.predict_proba(X_test)[:, 1] / N_SPLITS

    # LightGBM (fewer trees)
    model_lgb = LGBMClassifier(
        n_estimators=800,        # was 3000
        learning_rate=0.05,      # a bit higher since fewer trees
        num_leaves=48,
        max_depth=-1,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary",
        random_state=42,
        n_jobs=-1
    )
    model_lgb.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric="auc",
        callbacks=[]
    )
    val_pred_lgb = model_lgb.predict_proba(X_val)[:, 1]
    oof_lgb[val_idx] = val_pred_lgb
    preds_lgb += model_lgb.predict_proba(X_test)[:, 1] / N_SPLITS

    # XGBoost (fewer trees)
    model_xgb = XGBClassifier(
        n_estimators=800,        # was 3000
        learning_rate=0.05,
        max_depth=5,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",      # fast CPU histogram
        random_state=42,
        n_jobs=-1
    )
    model_xgb.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        verbose=False
    )
    val_pred_xgb = model_xgb.predict_proba(X_val)[:, 1]
    oof_xgb[val_idx] = val_pred_xgb
    preds_xgb += model_xgb.predict_proba(X_test)[:, 1] / N_SPLITS

# 8. Check individual model CV AUC
auc_cat = roc_auc_score(y_array, oof_cat)
auc_lgb = roc_auc_score(y_array, oof_lgb)
auc_xgb = roc_auc_score(y_array, oof_xgb)

print(f"CatBoost OOF AUC: {auc_cat:.6f}")
print(f"LightGBM OOF AUC: {auc_lgb:.6f}")
print(f"XGBoost OOF AUC: {auc_xgb:.6f}")


In [ ]:
# 9. Simple weighted blending - adjust weights if you see a clear winner in AUC
w_cat = 0.4
w_lgb = 0.3
w_xgb = 0.3

oof_blend = w_cat * oof_cat + w_lgb * oof_lgb + w_xgb * oof_xgb
auc_blend = roc_auc_score(y_array, oof_blend)
print(f"Blended OOF AUC: {auc_blend:.6f}")

test_pred_blend = w_cat * preds_cat + w_lgb * preds_lgb + w_xgb * preds_xgb


In [ ]:
# 10. Create submission file
submission = pd.DataFrame({
    ID_COL: test_id,
    TARGET: test_pred_blend
})

submission.to_csv("submission.csv", index=False)
submission.head()


In [ ]:
print(submission[TARGET].describe())
print(submission.isna().sum())
